In [2]:
import tkinter as tk
from spellchecker import SpellChecker
import re
import json
import os

spell = SpellChecker()

# -------------------------------
# USER DICTIONARY
# -------------------------------
USER_DICT_FILE = "user_dict.json"

if os.path.exists(USER_DICT_FILE):
    with open(USER_DICT_FILE, "r") as f:
        user_dictionary = set(json.load(f))
else:
    user_dictionary = set()

def save_user_dict():
    with open(USER_DICT_FILE, "w") as f:
        json.dump(list(user_dictionary), f)

# -------------------------------
# CUSTOM SHORTCUTS
# -------------------------------
custom_dict = {
    "shd": "should",
    "wud": "would",
    "cd": "could",
    "u": "you",
    "r": "are"
}

# -------------------------------
# NEXT WORD MODEL
# -------------------------------
next_word_model = {
    "i": ["am", "will", "have"],
    "you": ["are", "should", "can"],
    "my": ["name", "friend"],
    "hello": ["there"]
}

fallback_words = ["the", "is", "to", "and", "a"]

# -------------------------------
# UTILITIES
# -------------------------------
def get_last_word(widget):
    try:
        return widget.get("insert -1c wordstart", "insert -1c wordend")
    except:
        return ""

def normalize_case(word):
    return word.lower() if not (word.islower() or word.isupper()) else word

def remove_repeated_letters(word):
    return re.sub(r'(.)\1+', r'\1', word)

# -------------------------------
# 🔥 CAPITALIZE FIRST LETTER
# -------------------------------
def capitalize_first_letter(text_widget):
    try:
        content = text_widget.get("1.0", tk.END)

        if len(content.strip()) >= 1:
            first_char = content[0]

            if first_char.islower():
                text_widget.delete("1.0", "1.1")
                text_widget.insert("1.0", first_char.upper())

    except:
        pass

# -------------------------------
# GRAMMAR CORRECTION
# -------------------------------
def basic_grammar_fix(text):
    text = re.sub(r'\bi is\b', 'I am', text)
    text = re.sub(r'\byou is\b', 'you are', text)
    text = re.sub(r'\bhe go\b', 'he goes', text)
    text = re.sub(r'\bshe go\b', 'she goes', text)
    text = re.sub(r'\bit go\b', 'it goes', text)

    text = re.sub(r'\ba ([aeiouAEIOU])', r'an \1', text)
    text = re.sub(r'\b(\w+) \1\b', r'\1', text)

    return text

# -------------------------------
# SAFE AUTOCORRECT
# -------------------------------
def correct_word(word):
    w = word.lower()

    if w in user_dictionary:
        return word

    if w in custom_dict:
        return custom_dict[w]

    word_clean = normalize_case(remove_repeated_letters(word))
    w = word_clean.lower()

    if w in spell:
        return word_clean

    candidates = spell.candidates(w) or set()

    if candidates:
        def score(x):
            return (
                abs(len(x) - len(w)),
                x[0] != w[0],
            )

        best = min(candidates, key=score)

        if abs(len(best) - len(w)) <= 2:
            return best

    return word

# -------------------------------
# MAIN APP
# -------------------------------
class SmartKeyboard:
    def __init__(self, root):
        self.root = root
        self.root.title("Autocorrect Keyboard System")

        # TOP BAR
        self.next_bar = tk.Frame(root, bg="#e0e0e0")
        self.next_bar.pack(fill="x")

        self.next_buttons = []

        for i in range(3):
            self.next_bar.columnconfigure(i, weight=1)

            btn = tk.Button(
                self.next_bar,
                text="",
                font=("Arial", 12),
                relief="ridge",
                bg="white",
                command=lambda i=i: self.select_suggestion(i)
            )

            btn.grid(row=0, column=i, sticky="nsew", padx=3, pady=3)
            self.next_buttons.append(btn)

        # TEXT AREA
        self.text = tk.Text(root, font=("Arial", 14), wrap="word")
        self.text.pack(expand=True, fill="both")

        self.after_id = None

        self.text.bind("<KeyRelease>", self.on_typing)
        self.text.bind("<space>", self.on_space)

    # ---------------------------
    # REALTIME HANDLER
    # ---------------------------
    def on_typing(self, event):

        # 🔥 FIRST LETTER CAPITALIZATION
        capitalize_first_letter(self.text)

        if event.keysym == "space":
            return

        if self.after_id:
            self.root.after_cancel(self.after_id)

        self.after_id = self.root.after(120, self.handle_realtime)

    def handle_realtime(self):
        self.realtime_autocorrect()
        self.update_typing_suggestions()

    # ---------------------------
    # REALTIME AUTOCORRECT
    # ---------------------------
    def realtime_autocorrect(self):
        word = self.get_word_before_cursor()

        if len(word) < 4:
            return

        if len(word) >= 2 and word[-1] == word[-2]:
            return

        corrected = correct_word(word)

        if corrected != word:
            if corrected[0] == word[0] and abs(len(corrected) - len(word)) <= 1:

                start = self.text.index("insert wordstart")
                end = self.text.index("insert wordend")

                self.text.delete(start, end)
                self.text.insert(start, corrected)

    # ---------------------------
    # SUGGESTIONS
    # ---------------------------
    def update_typing_suggestions(self):
        word = self.get_word_before_cursor().lower()

        if len(word) < 2:
            self.clear_buttons()
            return

        candidates = spell.candidates(word) or set()

        suggestions = []

        for w in candidates:
            if w.startswith(word[0]) and abs(len(w) - len(word)) <= 2:
                suggestions.append(w)

        if not suggestions:
            suggestions = list(candidates)

        suggestions = sorted(
            suggestions,
            key=lambda x: abs(len(x) - len(word))
        )[:3]

        self.update_buttons(suggestions)

    # ---------------------------
    # SPACE EVENT
    # ---------------------------
    def on_space(self, event):

        word = get_last_word(self.text)
        corrected = correct_word(word)

        if corrected != word and len(word) > 2:

            start = self.text.index("insert -1c wordstart")
            end = self.text.index("insert -1c wordend")

            self.text.delete(start, end)
            self.text.insert(start, corrected)

        self.update_next_word_bar()

        # GRAMMAR FIX
        content = self.text.get("1.0", tk.END)
        new_content = basic_grammar_fix(content)

        if content != new_content:
            pos = self.text.index(tk.INSERT)

            self.text.delete("1.0", tk.END)
            self.text.insert("1.0", new_content)

            self.text.mark_set(tk.INSERT, pos)

        return None

    # ---------------------------
    # NEXT WORD
    # ---------------------------
    def update_next_word_bar(self):

        words = self.text.get("1.0", tk.END).strip().split()

        last = words[-1].lower() if words else ""

        predictions = next_word_model.get(last, [])

        if not predictions:
            predictions = fallback_words

        self.update_buttons(predictions[:3])

    # ---------------------------
    # BUTTON CONTROL
    # ---------------------------
    def update_buttons(self, words):

        for i, btn in enumerate(self.next_buttons):
            btn.config(text=words[i] if i < len(words) else "")

    def clear_buttons(self):

        for btn in self.next_buttons:
            btn.config(text="")

    def select_suggestion(self, index):

        word = self.next_buttons[index].cget("text")

        if not word:
            return

        start = self.text.index("insert wordstart")
        end = self.text.index("insert wordend")

        self.text.delete(start, end)
        self.text.insert(start, word + " ")

        user_dictionary.add(word.lower())
        save_user_dict()

    # ---------------------------
    # HELPER
    # ---------------------------
    def get_word_before_cursor(self):

        text = self.text.get("insert linestart", "insert")
        words = text.split()

        return words[-1] if words else ""

# -------------------------------
# RUN
# -------------------------------
root = tk.Tk()

app = SmartKeyboard(root)

root.mainloop()